# ChCODH2 좌표계 — 상동체 잔기를 1~636 번호로

ChCODH2(Q9F8A8, 636 aa)를 inverse folding 으로 재설계하려면 **어느 잔기를 건드려도 되는지**
를 알아야 하고, 그것은 근연 상동체의 보존도로 판단한다. 그 보존도를 계산하려면 먼저
**상동체의 잔기가 ChCODH2 의 몇 번 자리인지** 가 정해져 있어야 한다. 이 노트북이 그것만 한다.

`chcodh2_alignment.tsv` 의 `qaln`/`taln` 을 파싱해 상동체 1,827개의 잔기를 전부
**ChCODH2 서열 번호 1~636** 으로 옮긴 행렬을 만든다.

## 이것이 파이프라인의 척추다

여기가 틀리면 뒤의 보존도·Fix/Bias 판정이 **전부 조용히 틀린다.** 그래서 이 노트북의
본체는 산출물이 아니라 **3절의 검증**이다. 가정이 깨지면 경고가 아니라 예외다.

| | 검증 | 무엇을 잡는가 |
|---|---|---|
| **V1** | `qaln` 갭 제거 == 참조서열`[qstart..]` 부분서열 | 1-based/0-based 오프셋 오류. 전 행 정확히 일치해야 한다 |
| **V2** | `len(qaln) == len(taln)` | 정렬 문자열 쌍이 깨진 행. **HSP 선택 전 전 행**에 적용 |
| **V3** | 자기 히트의 `q_pos == t_pos` 가 636개 전부 | 좌표계를 독립적으로 재는 유일한 그물 |
| **V4** | base 자동 판별 — 0-based 는 **색인 0 이 실제로 있을 때만** | 질의가 참조와 1잔기 어긋난 판. `q_pos` 는 맞고 `t_pos` 만 +1 밀려 V1 을 통과한다 — **가장 위험한 경로** |
| **V5** | `qlen` 컬럼이 있으면 636 과 대조 | search 질의가 정말 이 서열이었나 (가장 싼 직접 증거) |
| **V6** | `taln` 소문자(소프트 마스킹) 검출 | V1~V3 를 전부 통과하면서 보존도만 과소계산되는 입력 |
| **V7** | universe 축소 검출 | 1,827행이 조용히 줄었는지 |

**V4 가 왜 따로 있는가** — 질의 서열이 참조와 1잔기만 어긋나면(개시 Met 유무·태그·다른
아이소폼) 0-based 로 오진되고, 그때 `q_pos` 는 맞게 나와 **V1 이 통과한다.** 자기 히트가
없으면 V3 도 못 하므로 그물이 하나도 남지 않는다. 그래서 0-based 를 채택하려면 파일
어딘가에 **색인 0 이 실제로 존재**해야 하고, 없으면 멈춘다.

## 2026-10-26 서버 실행에서 확인된 사실 (추측 아님)

- `chcodh2_alignment.tsv` — **헤더 있음**(mmseqs 필드 어휘 16/16), 1,827행 → target 1,827개,
  **HSP 중복 0건**, 절단 없음(universe 피복 1827/1827).
- `fident` 척도는 fraction(0~1), 결측 0건, 범위 0.2470~1.0000, 중앙 0.379.
  **자기 히트 fident 정확히 1.0000** 이고 `>=0.999` 인 target 은 그 1건뿐.
- 복원된 명령 — `mmseqs easy-search chcodh2.fasta ipr010047_all.fasta ... -s 7.5 -e 10000
  --max-seqs 100000 --format-output "query,target,fident,alnlen,qcov,tcov,evalue,bits,
  qstart,qend,tstart,tend,qaln,taln"`
- 질의 서열은 `blastp/chcodh2/chcodh2.fasta` (헤더 `>chcodh2|pdb|1SU6|A`, 636 aa) 이고
  **UniProt Q9F8A8 과 100% 일치**가 확인됐다. 2·3절이 그 사실을 파일로 다시 확인한다.
- 티어 경계(02 번에서 확정) — T1 `fident>=0.60`(31) / T2 `>=0.50`(129) / T3 `>=0.40`(760) /
  T4 전체(1,827). 0.50 아래에서 2Fe-2S(SF2)가 섞이기 시작하는 것이 경계 근거다.

## 번호 체계 — master 의 `r*` 가 아니라 참조 서열이 기준이다

`master_1827.tsv` 의 `r612` 가 논문 **V610** 과 +2 어긋난다고 보고됐다. 질의가 `pdb|1SU6|A`
였으므로 PDB 번호가 섞였을 가능성이 있다. 이 노트북의 좌표는 **참조 FASTA(Met1=1, 636 aa)
에서 직접 확인된 번호**이고, master 의 `r*` 는 7절에서 **잔기 값으로** 대조만 한다 —
컬럼 이름이 아니라 **값이 일치하는 쪽**이 답이다.

## 이 노트북이 하지 않는 것

- `usable`/`excluded` 로 **걸러내지 않는다.** 플래그만 실어 나르고 걸러낼지는 04 번이 정한다.
  **1,827행은 끝까지 1,827행이다.**
- mmseqs 를 돌리지 않는다. 기존 정렬 파일만 읽는다.
- 보존도를 계산하지 않는다. 그 **분모**(위치별 커버리지)까지만 만든다.

## 1. 설정

경로와 상수를 전부 여기 모았다. 환경이 바뀌면 이 두 셀만 고치고 1절부터 다시 실행한다.
**셀이 두 개인 이유** — 앞은 경로와 모듈(01·02 와 같은 꼴), 뒤는 이번 분석의 손잡이다.

`REFFA` 가 **좌표계의 유일한 출처**다. `input/` 에 없으면 2절이 원본에서 링크를 거는데,
원본 위치를 못 찾으면 거기서 멈춘다 — 그때는 질의로 쓴 636 aa FASTA 를 `input/` 에
직접 넣거나 아래 `REF_SRC` 후보에 실제 경로를 더할 것. **참조서열 없이는 번호가 없다.**

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd

# ── 고칠 곳은 여기뿐이다 ────────────────────────────────────────────
BASE   = Path("/mnt/af2results/mingyu/workspace/ipr010047_mmseqs")
RUN    = "20261025_01_clustering"   # 01·02 와 같은 묶음. script/ 와 result/ 가 이걸로 묶인다
NEEDS  = "2026-10-26a"              # 같이 받은 chcodh2_sweep.py 의 버전. 낡으면 멈춘다

SCRIPT = BASE / "script" / RUN            # 이 노트북과 chcodh2_sweep.py 가 같이 있는 곳
sys.path.insert(0, str(SCRIPT))
try:
    import chcodh2_sweep as cs
except ModuleNotFoundError:
    raise SystemExit(f"{SCRIPT}/chcodh2_sweep.py 가 없다 — 노트북과 같은 폴더에 두고 다시 실행할 것.")
if cs.VERSION < NEEDS:   # 함수가 추가되기만 하므로 최신 한 벌로 모든 노트북이 돈다
    raise SystemExit(f"라이브러리가 낡았다: {cs.VERSION} < 필요 {NEEDS}. 새 chcodh2_sweep.py 를 받을 것.")

INPUT, OUTDIR = BASE / "input", BASE / "result" / RUN    # 입력은 공용, 결과는 묶음별
FASTA, MASTER = INPUT / "protein-matching-IPR010047.fasta", INPUT / "master_1827.tsv"
ALIGN, REFFA = INPUT / "chcodh2_alignment.tsv", INPUT / "chcodh2.fasta"
for d in (INPUT, OUTDIR): d.mkdir(parents=True, exist_ok=True)
print(f"묶음 {RUN}\n모듈 {cs.__file__} (v{cs.VERSION})\n입력 {INPUT}\n결과 {OUTDIR}")

이쪽이 **이번 분석의 손잡이**다.

- `ASSUME_BASE = None` — base 를 **데이터로 판정한다**(V4). `0` 을 넣는 것은 "파일이 진짜
  0-based 임을 사람이 확인했다" 는 선언이며, 그 선언이 틀리면 `t_pos` 가 전 행 밀린 채
  조용히 틀린다. **3절의 색인 0 증거를 보고 나서만 건드릴 것.**
- `UPPER_ALN = False` — `taln` 에 소문자가 있으면 3절에서 예외가 난다(V6). 그것이 실제로
  소프트 마스킹이라고 확인됐을 때만 `True` 로 바꾼다.
- `SAVE_LONG = False` — `equiv_map.tsv`(long, 100만행급)는 기본으로 끈다. **보존도는 행렬만
  으로 계산된다.** 04 번이 상동체 쪽 번호(`t_pos`)를 필요로 하면 `True` 로 켠다.
- `TIERS` — 02 번에서 확정된 티어 경계. 8절이 각 티어의 위치별 분모를 센다.
- `KNOWN` — 2026-10-26 서버 실행값. 2·3·4·8절이 자동 대조해 어긋나면 띄운다.

In [ ]:
REF_ACC = cs.CHCODH2_ACC          # Q9F8A8 — 좌표의 주인. 자기 히트가 V3 의 그물이다
ALIGN_COLUMNS = None              # 열 이름이 안 맞으면 cs.ALIGN_FORMAT_OUTPUT 를 넣을 것
ASSUME_BASE   = None              # None = 데이터로 판정(V4). 0 은 '사람이 확인했다' 는 선언이다
UPPER_ALN     = False             # taln 소문자가 실제로 소프트 마스킹일 때만 True (V6)
SAVE_LONG     = False             # equiv_map.tsv(long) 저장 여부. 행렬만으로 보존도는 계산된다
TIERS = {"T1": 0.60, "T2": 0.50, "T3": 0.40, "T4": 0.0}   # 02 번에서 확정된 티어 경계

# 2026-10-26 서버 실행에서 확인된 값. 어긋나면 입력이 달라진 것이다 (멈추지는 않고 띄운다)
KNOWN = dict(n_seq=1827, n_align_rows=1827, n_merged_hsp=0, fident_median=0.379,
             n_tier={"T1": 31, "T2": 129, "T3": 760, "T4": 1827})
PAPER_POSITIONS = sorted(cs.KNOWN_POSITIONS)      # 논문 위치 16개 (3절에서 참조서열로 재확인)

# 입력 원본의 현재 위치. 2절이 input/ 에 없으면 링크를 건다 (원본은 읽기만 한다)
MNT, DOCK = Path("/mnt/af2results/mingyu"), Path("/mnt/af2results/mingyu/workspace/docking_simulation")
REF_SRC = next((p for p in (SCRIPT / REFFA.name, DOCK / "blastp/chcodh2" / REFFA.name,
                            MNT / "blastp/chcodh2" / REFFA.name, MNT / "chcodh2" / REFFA.name)
                if p.exists()), None)
SRC = {FASTA.name: MNT / "chcodh2" / FASTA.name,
       MASTER.name: DOCK / "result/homolog_split" / MASTER.name,
       ALIGN.name: DOCK / "result/homolog_selection" / ALIGN.name,
       REFFA.name: REF_SRC}
print(f"티어 {TIERS}\n논문 위치 {PAPER_POSITIONS}\n참조 FASTA 원본 {REF_SRC}")

## 2. 입력 감사 — 숫자를 눈으로 본다

02 번의 `read_alignment` 를 그대로 재사용한다(열 이름 출처 판정·HSP 병합·`fident` 척도
정규화). **여기서 볼 것**은 위에 적힌 2026-10-26 실행값과 어긋나는 데가 있는지다.

1. **FASTA** — 1,827 레코드인가. `Q9F8A8` 가 들어 있는가 (없으면 멈춘다).
2. **alignment** — 열 이름을 **무엇을 근거로** 정했는지(헤더가 있어야 한다), 행 1,827 →
   target 1,827 에 **HSP 병합 0건**인지, `fident` 가 fraction 인지, 중앙값이 0.379 인지,
   universe 를 다 덮는지(`--max-seqs 100000 -e 10000` 이었으므로 절단은 없어야 한다).
3. **`assert_alignment_usable`** — 열이 한 칸 밀려 `fident` 자리에 `qcov`(0.95)나
   `qstart`(1)가 들어와도 0~1 범위라 척도 판정을 그냥 통과한다. 자기 히트 1.0 ·
   상수 아님 · universe 밖 없음 · `>=0.999` 가 1건뿐 — 네 가지로 못 박는다.

이 셀은 **좌표를 만들지 않는다.** 정렬 파일이 그 파일인지만 확인한다.

**주의** — `assert_alignment_usable` 은 **자기 히트가 있다는 것을 전제로** 한다(열이 밀렸는지
자기 히트 1.0 으로 재기 때문이다). 2026-10-26 입력에는 자기 히트가 있으므로 통과한다.
언젠가 자기 히트 없는 정렬로 돌려야 하면 이 셀이 멈추는데, 그때는 **V3 그물까지 같이
사라진다**는 뜻이다. 마지막 두 줄을 지우고 3절로 갈 수는 있지만, 그러면 base 판정을
사람이 직접 확인해야 하고 그 사실을 보고서에 적어야 한다.

In [ ]:
print("\n".join(cs.link_inputs([FASTA, MASTER, ALIGN, REFFA], SRC)))

universe, uni = cs.fasta_universe(FASTA)
mark = "" if uni["n_unique_accession"] == KNOWN["n_seq"] else "  ⚠ 2026-10-26 실행과 어긋난다"
print(f"\nFASTA 레코드 {uni['n_records']} / 고유 accession {uni['n_unique_accession']} "
      f"(기대 {KNOWN['n_seq']}), 중복 {uni['n_duplicate_accession']}{mark}")
if REF_ACC not in universe:
    raise SystemExit(f"{REF_ACC} 가 FASTA 에 없다 — 좌표의 주인이 빠진 universe 는 의미가 없다.")

align, am = cs.read_alignment(ALIGN, columns=ALIGN_COLUMNS, ref_acc=REF_ACC)
acov = cs.alignment_coverage(align, universe)
print(f"\n정렬 열 이름 출처: {am['column_source']}  (헤더 판정: {am['header_reason']})")
print(f"  행 {am['n_rows']} (기대 {KNOWN['n_align_rows']}) → target {am['n_unique_target']}개, "
      f"HSP 병합 {am['n_merged']}건 (기대 {KNOWN['n_merged_hsp']})"
      + ("" if am["n_merged"] == KNOWN["n_merged_hsp"] else "  ⚠ 어긋난다 — HSP 선택 규칙이 작동한다"))
print(f"  fident {am['fident_scale']}, 결측 {am['n_fident_missing']}건, "
      f"범위 {am['fident_min']:.4f}~{am['fident_max']:.4f}, 중앙 {am['fident_median']:.4f} "
      f"(기대 {KNOWN['fident_median']})"
      + ("" if abs(am["fident_median"] - KNOWN["fident_median"]) < 0.01 else "  ⚠ 입력이 다를 수 있다"))
print(f"  universe 피복 {acov['n_in_universe']}/{acov['n_universe']}, 결측 {acov['n_missing']}"
      + (f" 예 {acov['missing_examples'][:5]} ← 절단 의심" if acov["n_missing"] else " ← 절단 없음"))
for line in cs.assert_alignment_usable(align, am, acov, ref_acc=REF_ACC):
    print("  " + line)

## 3. 좌표계 구축 + 검증 V1~V7  ★이 노트북의 본체★

순서가 중요하다 — **V2(전 행) → V5 → V6 → HSP 선택 → V4/V1 → 매핑(행마다 V1 재확인) → V3.**
`[V?] 통과` 가 한 줄씩 찍힌다. **생략**은 실패가 아니라 "그 그물이 없다" 는 뜻이므로
(자기 히트가 없거나 `qlen` 컬럼이 없을 때) 그만큼 base 오판에 취약해진다는 신호다.

**무엇을 보고 판단하는가**

1. **참조서열** — 636 aa 인가. 논문 16개 위치의 잔기가 참조서열과 전부 맞는가
   (`F41 E43 R57 N59 … V610`). 이것이 **번호 체계가 맞다는 유일한 직접 증거**다.
   accession 이 `Q9F8A8` 가 아니어도(질의 헤더가 `pdb|1SU6|A` 였다) 서열 자체가 맞으면 된다.
2. **채택한 base 와 그 근거** — 1-based 로 몇 행이 맞았고 0-based 로 몇 행이 맞았는지,
   그리고 **색인 0 이 파일에 실제로 있는지**(`qstart==0`/`tstart==0` 행 수). 0 이 없는데
   0-based 가 이기면 그것은 base 문제가 아니라 **질의 서열이 참조와 어긋난 것**이므로 멈춘다.
3. **자기 히트** — `q_pos == t_pos` 가 636/636 이고 잔기도 636/636 이면, 좌표계가 참조서열과
   같다는 것이 **정렬 파일 자신에 의해** 확인된 것이다.

In [ ]:
REF_SEQ, refm = cs.read_reference_seq(REFFA, ref_acc=REF_ACC)
print(f"참조서열 {refm['length']} aa — '{refm['header'][:70]}'")
print(f"  accession '{refm['accession']}' == {REF_ACC}: {refm['accession_matches_reference']}"
      + ("" if refm["accession_matches_reference"] else "  ← pdb 헤더면 정상. 서열 자체를 아래에서 확인한다"))
kp = cs.verify_known_positions(REF_SEQ)
print(f"  논문 {kp['n_positions']}개 위치 잔기 일치: {kp['passed']}  {kp['residues']}")

rows, rm = cs.read_alignment_rows(ALIGN, columns=ALIGN_COLUMNS, ref_acc=REF_ACC)
print(f"\n정렬을 행 그대로 {rm['n_rows']}행 / target {rm['n_unique_target']}개 읽었다 "
      f"(열 이름 출처 {rm['column_source']}, qlen 컬럼 {'있다' if 'qlen' in rows.columns else '없다'})")

equiv, em = cs.parse_equiv(rows, REF_SEQ, assume_base=ASSUME_BASE, ref_acc=REF_ACC,
                           upper_aln=UPPER_ALN)
print("\n■ 검증 — 하나라도 깨지면 위에서 예외가 났다. 여기까지 왔으면 아래가 통과 기록이다")
print("\n".join(cs.equiv_check_lines(em)))

b = [c for c in em["checks"] if c["check"] == "V4"][0]
print(f"\n■ 채택한 base = {em['base']}")
print(f"  근거: {em['base_basis']}")
print(f"  색인 0 증거 — qstart 최소 {b['qstart_min']}, tstart 최소 {b['tstart_min']}, "
      f"qstart==0 {b['n_qstart_zero']}행, tstart==0 {b['n_tstart_zero']}행 → "
      f"0-based 채택 가능 {b['zero_index_present']}")
print(f"\n대응 {em['n_pairs']:,}개 (잔기 {em['n_residue_pairs']:,} / 갭 {em['n_gap_pairs']:,}), "
      f"HSP 병합 {em['hsp']['n_merged']}건, 삽입 열 {em['insertion_cols']:,}개는 "
      f"ChCODH2 번호가 없어 버렸다")

## 4. 행렬 — 1,827 x 636

행=accession(**1,827 전부**), 열=1..636, 값=그 자리에 정렬된 상동체 잔기.
정렬에 아예 들어가지 않은 서열도 **행으로 남는다**(값은 전부 `-`). 규칙은 하나다 —
**1,827행은 끝까지 1,827행.** 행 수가 그보다 적으면 V7 로 띄운다.

`-` 은 두 가지를 겸한다 — 정렬 안에서의 갭(상동체에 그 잔기가 없다)과 정렬 블록 밖
(애초에 정렬되지 않았다). 파일을 단순하게 두려고 같은 문자를 쓰지만, **둘을 구분해야 하는
계산은 행렬에 같이 실려 오는 커버리지 마스크를 쓴다**(5절이 그렇게 한다). tsv 로 저장한
행렬을 다시 읽으면 그 마스크가 없어 커버리지를 계산할 수 없다 — 그때는 예외가 난다.

마지막 줄이 **V3 를 행렬로 다시 보는 것**이다. 자기 히트 행을 이어 붙이면 참조서열과
글자 하나까지 같아야 한다.

In [ ]:
mat = cs.build_residue_matrix(equiv, universe, REF_SEQ)
mm = mat.attrs["meta"]
cells = mm["n_rows"] * mm["n_position_columns"]
print(f"행렬 {mat.shape[0]} x {mat.shape[1]}  (기대 {KNOWN['n_seq']} x {len(REF_SEQ)})"
      + ("" if mm["n_rows"] == KNOWN["n_seq"] else "  ⚠ [V7] 행 수가 어긋난다"))
print(f"  정렬에 들어간 서열 {mm['n_in_alignment']} / 안 들어간 서열 {mm['n_not_in_alignment']} "
      f"← 행은 남는다 (04 번이 usable/excluded 로 정할 몫이다)")
print(f"  덮인 칸 {mm['n_covered_cells']:,}/{cells:,} ({mm['n_covered_cells'] / cells:.3f}), "
      f"그중 갭 {mm['n_gap_cells']:,}개")
print(f"  메모리 {mm['memory_bytes'] / (1 << 20):.1f} MiB (값 + 커버리지 마스크)")
for note in mm["notes"]:
    print("  ⚠ " + note)
vals = mat.to_numpy(dtype="<U1")
print(f"  빈 칸 {int((vals == '').sum())}개 — 0 이어야 한다 "
      f"(정렬 밖은 '{mm['outside_char']}', 결측이라는 개념이 없다)")

print("\n■ 앞 5행 10열 (값 = 그 ChCODH2 위치에 정렬된 상동체 잔기)")
print(mat.iloc[:5, :10].to_string())
v3 = [c for c in em["checks"] if c["check"] == "V3"][0]
same = ("".join(mat.loc[REF_ACC].astype(str)) == REF_SEQ) if v3["present"] else None
print(f"\n자기 히트 행 == 참조서열: {same}  ← V3 와 같은 말을 행렬로 다시 본다 "
      f"(None = 자기 히트가 없어 재확인 불가 — 3절에서 V3 가 생략됐다)")
if same is False:
    raise SystemExit("자기 히트 행이 참조서열과 다르다 — 행렬 투영이 틀렸다. 중단한다.")

## 5. 위치 커버리지 — 636행, 보존도의 분모

위치마다 `n_residue`(잔기) / `n_gap_within_aln`(정렬 안 갭) / `n_outside_aln`(정렬 밖) /
`n_identical`(참조와 같은 잔기)을 센다. 네 수와 `n_not_in_alignment` 의 합이 모든 위치에서
행 수와 같아야 하고, 깨지면 예외다 — **분모가 틀리면 보존도가 조용히 틀린다.**

**목적은 보존도를 믿을 수 없는 위치를 미리 아는 것이다.** 상동체가 거의 정렬되지 않는
구간(말단·삽입 영역)에서는 분모가 작아 보존도가 요동친다. 커버리지 하위 10개와,
`universe` 의 절반도 못 덮는 **연속 구간**을 찍는다. 마지막 표가 **논문 16개 위치의 분모**다 —
이 숫자가 04 번에서 그 자리의 보존도를 믿을지 말지를 정한다.

In [ ]:
cov = cs.position_coverage(mat, REF_SEQ)
csum = cs.coverage_summary(cov, worst=10, low_frac=0.5)
print(f"위치 {csum['n_positions']}개 · 행 {csum['n_rows']} · 정렬 포함 {csum['n_in_alignment']}")
print(f"  n_residue 최소 {csum['min_n_residue']} / 중앙 {csum['median_n_residue']:.0f} / "
      f"최대 {csum['max_n_residue']},  잔기가 0 인 위치 {csum['n_positions_zero_residue']}개")
print(f"  universe 의 절반도 못 덮는 위치 {csum['n_positions_below_low_frac']}개 — "
      f"연속 구간(1-based) {csum['low_runs_1based'][:8]}")
print("  ↑ 이 구간의 보존도는 분모가 작아 믿을 수 없다. 04 번에서 따로 표시할 것")

print("\n■ 커버리지가 가장 낮은 10개 위치")
print(pd.DataFrame(csum["worst_positions"]).to_string(index=False))

print(f"\n■ 논문 16개 위치의 분모 (최소 {csum['known_min_n_residue']}, "
      f"중앙 {csum['known_median_n_residue']:.0f})")
print(cov[cov["is_known_position"]][["q_pos", "q_aa", "known_role", "n_residue",
      "n_gap_within_aln", "n_outside_aln", "n_identical",
      "identity_frac_of_residues"]].round(3).to_string(index=False))

## 6. 논문 16개 위치의 상동체 잔기

행=accession, 열=논문 위치. 값은 세 가지를 구분한다 — **잔기** / `-`(정렬 안 갭) /
`''`(정렬 블록 밖). 그 뒤에 위치별 빈도표가 붙는다.

**여기서 눈으로 확인되는 것**

- `F41` 이 정말 보존되는가. `identity_frac` 가 1.0 에 가까우면 **건드리면 안 되는 자리**다
  (F41A 가 활성을 잃는다는 논문과 같은 말이어야 한다).
- `A559` 자리가 다른 CODH 에선 **W/Y** 인가. 그렇다면 `A559W` 는 "자연에 없는 설계" 가
  아니라 **다른 CODH 가 이미 쓰는 잔기**이고, O2 내성 61~148배의 근거가 보존도로 뒷받침된다.
- `n_distinct_residue` 가 크고 `identity_frac` 가 낮은 자리가 **바꿔도 되는 후보**다.

`n_residue` 가 작은 위치(5절의 분모)는 빈도표의 비율도 믿을 수 없다. 두 표를 같이 볼 것.

In [ ]:
known = cs.known_position_equivalents(equiv, PAPER_POSITIONS, universe=universe, ref_seq=REF_SEQ)
freq = cs.known_position_frequency(known, REF_SEQ, top=5)
print(f"논문 위치 표 {known.shape[0]}행 x {known.shape[1]}열 "
      f"(값: 잔기 / '-' 정렬 안 갭 / '' 정렬 밖)")

print("\n■ 위치별 요약 — identity_frac 가 낮고 n_distinct 가 큰 자리가 '바꿔도 되는' 후보다")
print(freq[["q_pos", "q_aa", "known_role", "n_residue", "n_gap", "n_outside",
            "n_identical", "identity_frac", "n_distinct_residue"]].round(3).to_string(index=False))

print("\n■ 각 자리에 실제로 나오는 잔기 (상위 5개, 괄호는 잔기 중 비율)")
for r in freq.itertuples():
    print(f"  {int(r.q_pos):>3}{r.q_aa} {str(r.known_role):<26} {r.top_residues}")

print("\n■ 앞 5행 미리보기")
print(known.head(5).to_string())

## 7. master 의 `r*` 컬럼 대조 — 이름이 아니라 **잔기 값**으로 판정한다

`master_1827.tsv` 에 `r41 r43 r44 r57 r59 r60 r61 r62 r583 r612` 가 있고, `r612` 가 논문의
**V610** 과 +2 어긋난다고 보고됐다. 이름만 보면 어느 쪽이 맞는지 알 수 없다. 그래서
**각 `r*` 컬럼의 값을 이 좌표계의 잔기와 비교해 일치율이 가장 높은 위치를 그 컬럼의
정체로 판정한다.**

후보는 이름 주변이 아니라 **참조서열 1~636 전부**다. 이름 ±4 만 보면 어긋남이 그 창보다
클 때(PDB 번호가 10 어긋나는 식) 창 안의 엉뚱한 이웃이 1등이 되어 **확신에 찬 오답**이
나온다. 참조서열에는 같은 잔기가 널려 있어서(600번과 610번이 모두 `V`) 그 일이 실제로
생긴다 — 창만 보면 "값은 610 이다" 를 0.38 짜리 근거로 단정해 버린다.

표에서 볼 것

- `frac` 이 1.0 에 가까운 위치가 **하나뿐**이면 결론은 분명하다. 1등이 하한(`min_frac`)에
  못 미치거나 **먼 2등**과 `margin` 만큼 벌어지지 않으면 **판정 불가**로 남는다.
  판정 불가는 "이름이 맞다" 가 아니다 — **그 컬럼이 어느 자리인지 모른다**는 뜻이므로
  04 번에서 쓰면 안 된다.
- `reference_row_equal` 은 **자기 자신(Q9F8A8) 행**이 그 위치의 참조 잔기와 같은지로,
  위치 610 은 `V`, 612 는 `L` 이므로 그 한 칸만으로도 갈린다. 다만 같은 잔기는 참조서열에
  여러 번 나오므로(`V` 는 수십 곳) 이것만으로 결론을 내지 말고 `frac` 과 같이 볼 것.

판정 결과가 어떻든 **이 노트북의 좌표는 바뀌지 않는다**(참조서열 기준). 바뀌는 것은
04 번이 master 의 `r*` 를 믿어도 되는지다. 결론은 `shifted_columns` 에 담긴다.

In [ ]:
master, mmeta = cs.read_master_table(MASTER)
mcmp = None
if master is None:
    print(f"master 없음 ({mmeta['reason']}) — r* 대조를 건너뛴다 (좌표계에는 영향 없음)")
else:
    print(f"master {mmeta['n_rows']}행, 위치 컬럼 {mmeta['position_columns']}")
    print("  usable/excluded 류 컬럼은 여기서 **걸러내지 않는다** — 04 번의 몫이다")
    mcmp = cs.compare_master_positions(master, equiv, ref_seq=REF_SEQ, ref_acc=REF_ACC)
    print(f"  공통 accession {mcmp['n_common_accession']}/{mcmp['n_master_rows']} · "
          f"후보는 참조서열 {mcmp['scanned_positions']} 위치 **전부** (1등 일치율 하한 "
          f"{mcmp['min_frac']}, 먼 2등과 {mcmp['margin']} 이상 벌어질 때만 판정한다)")
    print("\n".join(cs.master_compare_lines(mcmp, focus=("r612",))))
    print(f"\n결론 — 이름과 다른 위치였던 컬럼: {mcmp['shifted_columns']}")
    print(f"        판정 불가 {mcmp['n_undecidable']}개 · 사람이 정할 것 남음 {mcmp['needs_decision']}")
    print("  ↑ '판정 불가' 는 '이름이 맞다' 가 아니다 — 어느 자리인지 모른다는 뜻이다")

## 8. 티어별 커버리지 — 04 번의 전제

T1~T4 각각에서 **위치별로 몇 개의 서열이 실제로 정렬됐는지**를 센다. 티어가 좁아질수록
서열은 순수해지지만 분모가 작아진다. **T1(31개)에서 커버리지가 낮은 위치는 보존도를
낼 수 없다** — 31개 중 5개만 정렬된 자리의 "보존도 80%" 는 숫자가 아니라 잡음이다.

표에서 볼 것

- `n_known_below_frac` — 그 티어에서 분모가 `low_frac`(=0.8) 미만인 **논문 위치의 개수**.
  0 이 아니면 그 위치는 04 번에서 그 티어로 판정할 수 없다. 어느 위치인지 같이 찍힌다.
- 마지막 표가 **T1 에서 가장 분모가 작은 위치 10개**다. 04 번의 Fix/Bias 판정에서 이
  위치들은 "보존도 미정" 으로 남겨야 한다.
- 티어 크기가 02 번의 31/129/760/1827 과 다르면 입력이 달라진 것이므로 띄운다.
  (`n_effective` 로 중복을 보정하는 것은 04 번의 몫이다 — 여기서는 서열 수만 센다.)

In [ ]:
tiers = {k: (sorted(universe) if v <= 0
             else sorted(cs.identity_set(align, v, universe, ref_acc=REF_ACC)))
         for k, v in TIERS.items()}
for k, v in TIERS.items():
    exp = KNOWN["n_tier"].get(k)
    print(f"{k}  fident>={v:.2f}  n={len(tiers[k])}"
          + ("" if exp is None or len(tiers[k]) == exp else f"   ⚠ 02번은 {exp} 였다"))

tcov = cs.tier_coverage(mat, tiers, REF_SEQ)
tsum = cs.tier_coverage_summary(tcov, low_frac=0.8)
print("\n■ 티어 요약 (low_frac=0.8 — 그 티어 서열의 80% 미만만 정렬된 위치를 센다)")
print(tsum.to_string(index=False))

print("\n■ 논문 16개 위치 x 티어 — n_residue (= 보존도의 분모)")
piv = tcov[tcov["is_known_position"]].pivot(index="q_pos", columns="tier", values="n_residue")
piv.insert(0, "q_aa", [REF_SEQ[int(p) - 1] for p in piv.index])
print(piv.to_string())

print("\n■ T1 에서 분모가 가장 작은 위치 10개 — 04 번에서 '보존도 미정' 으로 남길 자리")
t1 = tcov[tcov["tier"] == "T1"].sort_values(["n_residue", "q_pos"], kind="mergesort").head(10)
print(t1[["q_pos", "q_aa", "is_known_position", "n_residue", "n_gap_within_aln",
          "n_outside_aln", "residue_frac_tier"]].round(3).to_string(index=False))

## 9. 그림 (선택)

matplotlib 이 있으면 두 장. 왼쪽은 **위치별 커버리지**(`n_residue`/행 수)로, 바닥에 가까운
구간이 5절에서 센 "믿을 수 없는 위치" 다. 오른쪽은 **위치별 동일성**
(`n_identical`/`n_residue`)이고, 빨간 점이 논문 16개 위치다 — 보존된 자리는 위쪽,
바꿔도 되는 자리는 아래쪽에 모인다.

**축 라벨은 영문이다** — matplotlib 에 한글 폰트가 없는 환경에서 한글이 □ 로 깨지기 때문이다.
matplotlib 이 없으면 같은 내용을 **50 위치 구간 표**로 찍는다. 5·6절 표로 판단은 충분히 된다.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib 이 없다 — 구간 표로 대신 찍는다 (5·6절 표로 판단은 충분하다)")
    bins = (cov.assign(bin50=((cov["q_pos"] - 1) // 50) * 50 + 1)
               .groupby("bin50")[["n_residue", "n_identical"]].mean().round(1))
    bins["identity_frac"] = (bins["n_identical"] / bins["n_residue"]).round(3)
    print(bins.to_string())

if plt is not None:                                 # 축 이름은 한글 글꼴 경고를 피해 영문
    kn = cov[cov["is_known_position"]]
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax[0].fill_between(cov["q_pos"], cov["residue_frac_universe"], step="mid", alpha=0.6)
    ax[0].plot(kn["q_pos"], kn["residue_frac_universe"], "r.", ms=8)
    ax[0].set_ylabel("residue fraction of universe")
    ax[0].set_title(f"coverage per position (red = {len(kn)} paper positions)")
    ax[1].plot(cov["q_pos"], cov["identity_frac_of_residues"], lw=0.7)
    ax[1].plot(kn["q_pos"], kn["identity_frac_of_residues"], "r.", ms=8)
    for _, r in kn.iterrows():
        ax[1].annotate(f"{int(r['q_pos'])}{r['q_aa']}", (r["q_pos"], r["identity_frac_of_residues"]),
                       fontsize=7, rotation=90, textcoords="offset points", xytext=(0, 4))
    ax[1].set_xlabel("ChCODH2 position (Met1 = 1)")
    ax[1].set_ylabel("identity among aligned residues")
    fig.tight_layout(); plt.show()

## 10. 저장

**파일 이름을 분리한다** — 한 폴더(`result/20261025_01_clustering/`)를 여러 노트북이
공유하므로 `params.json` 을 그대로 쓰면 02 번의 것을 덮어쓴다. 이 노트북은
`params_03_equivmap.json` 을 쓴다.

| 파일 | 내용 |
|---|---|
| `residue_matrix.tsv` | 1,827 x 636. **04 번이 읽을 본 파일.** 보존도는 이것만으로 계산된다 |
| `position_coverage.tsv` | 636행. 위치별 분모 (`n_residue`/`n_gap`/`n_outside`/`n_identical`) |
| `known_position_equivalents.tsv` | 행=accession, 열=논문 16개 위치 |
| `known_position_frequency.tsv` | 논문 위치별 잔기 빈도와 `identity_frac` |
| `tier_coverage.tsv` / `tier_coverage_summary.tsv` | 티어 x 위치 분모, 티어별 요약 |
| `equiv_per_accession.tsv` | 상동체별 정렬 구간(`q_from..q_to`, `t_from..t_to`)과 갭 수 |
| `equiv_audit.json` | 검증 V1~V7 의 통과 기록·채택한 base 와 근거·master 대조 결론 |
| `params_03_equivmap.json` | 입력 해시·버전·손잡이 값 |

`equiv_map.tsv`(long, 100만행급)는 `SAVE_LONG` 이 켜져 있을 때만 쓴다. 상동체 쪽 번호
(`t_pos`)가 필요한 단계가 생기면 1절에서 켤 것.

**다음(04 번)이 정할 것** — ① `usable`/`excluded` 를 걸러낼지 ② 어느 티어로 보존도를 낼지
③ 분모가 작은 위치를 어떻게 표시할지 ④ 7절 결론에 따라 master 의 `r*` 를 쓸지 버릴지.

**04 번이 이 파일들을 읽을 때** — `residue_matrix.tsv` 의 열 이름은 파일에서 읽으면
**문자열** `"1"`…`"636"` 이다(메모리에서는 정수였다). 위치로 조인하려면 한쪽을 맞춰야
한다(`df.columns = df.columns.astype(int)`). 그리고 `pd.read_csv(..., dtype=str,
keep_default_na=False)` 로 읽을 것 — 그냥 읽으면 `known_position_equivalents.tsv` 의
`''`(정렬 블록 밖)가 `NaN` 이 되고, 행렬의 `-` 는 괜찮지만 `N`/`NA` 같은 잔기 문자가
결측으로 바뀐다. **커버리지 마스크는 파일에 없다** — 분모가 필요하면 다시 만들지 말고
`position_coverage.tsv` 를 읽을 것.

In [ ]:
TSV = {"position_coverage.tsv": cov, "known_position_frequency.tsv": freq,
       "tier_coverage.tsv": tcov, "tier_coverage_summary.tsv": tsum,
       "equiv_per_accession.tsv": em["per_accession"]}
cs.write_residue_matrix(OUTDIR / "residue_matrix.tsv", mat)
for name, df in TSV.items():
    df.to_csv(OUTDIR / name, sep="\t", index=False)
known.to_csv(OUTDIR / "known_position_equivalents.tsv", sep="\t")   # 색인이 accession 이다
long_meta = cs.write_equiv_long(OUTDIR / "equiv_map.tsv", equiv) if SAVE_LONG else None
files = (["residue_matrix.tsv"] + list(TSV) + ["known_position_equivalents.tsv"]
         + (["equiv_map.tsv"] if SAVE_LONG else []))
audit = dict(reference=refm, known_positions=kp, fasta=uni, alignment=am, alignment_rows=rm,
             alignment_coverage=acov, matrix=mm, position_coverage=csum, master=mmeta,
             equiv={k: v for k, v in em.items() if k != "per_accession"},
             tiers={k: len(v) for k, v in tiers.items()},
             master_position_compare=mcmp, equiv_map_long=long_meta)
(OUTDIR/"equiv_audit.json").write_text(json.dumps(audit, indent=2, default=str))
cs.write_params(OUTDIR, "ChCODH2 좌표계 (notebook)", inputs=[FASTA, MASTER, ALIGN, REFFA],
                outputs=[OUTDIR / n for n in files],   # 폴더 glob 금지 — 01·02 것이 섞인다
                commands=["(mmseqs 미실행 — 기존 정렬 파일만 읽음)"],
                filename="params_03_equivmap.json", tiers=TIERS, n_pairs=em["n_pairs"],
                base=em["base"], base_basis=em["base_basis"], assume_base=ASSUME_BASE,
                upper_aln=UPPER_ALN, save_long=bool(SAVE_LONG), paper_positions=PAPER_POSITIONS,
                shifted_master_columns=(mcmp or {}).get("shifted_columns"))
print(f"저장 위치 {OUTDIR}\n" + "\n".join(
    f"   {n:34s} {(OUTDIR / n).stat().st_size:>12,} bytes" for n in files))